# 02 Data Preparation

Notebook ini adalah bagian Data Preparation untuk proyek Deteksi Aktivitas Deforestasi Hutan Tropis melalui Klasifikasi Multi-Label Citra Satelit (Kelompok 10). Isinya: data cleaning, feature engineering, transformasi data, splitting train validation test, keputusan data balancing, dan ringkasan Apa Mengapa Bagaimana untuk setiap langkah.

Pelaksana resmi persiapan data adalah skrip scripts/persiapan_data.py, yang menjalankan seluruh tahap secara otomatis dan bisa diulang dengan hasil yang sama (seed tetap 42). Notebook ini membaca keluaran skrip itu, memverifikasinya, dan menjelaskan alasan setiap langkah, sehingga notebook ini sendiri bisa dijalankan ulang dari atas ke bawah sesudah skrip dijalankan satu kali.

In [1]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
print("Root proyek:", ROOT)

Root proyek: /home/hatch/workspace/learning-problem-deforestasi


## 1. Data cleaning

Cleaning mengikuti aturan resmi dataset dan dijalankan skrip persiapan data. Ringkasannya dibaca dari cleaning_log.json yang ditulis skrip saat berjalan.

In [2]:
import json
import pandas as pd

log = json.loads((PROC / "cleaning_log.json").read_text(encoding="utf-8"))
print("Baris CSV awal            :", log["baris_csv_awal"])
print("Duplikat image_name       :", log["duplikat_image_name"]["jumlah_baris_dibuang"])
print("File tidak ada di disk    :", log["file_tidak_ada_di_disk"]["jumlah"])
print("Citra rusak               :", log["citra_rusak"]["jumlah"])
print("Cloudy label lain dihapus :", log["cloudy_label_lain_dihapus"]["jumlah"])
print("Dibuang cuaca ganda       :", log["dibuang_label_cuaca_ganda"]["jumlah"])
print("Dibuang tanpa label cuaca :", log["dibuang_tanpa_label_cuaca"]["jumlah"], log["dibuang_tanpa_label_cuaca"]["nama"])
print("Total data bersih         :", log["total_bersih"])
print("Metode split              :", log["metode_split"])
print("Deviasi proporsi maksimum :", log["split"]["deviasi_proporsi_label_maksimum"])

Baris CSV awal            : 40479
Duplikat image_name       : 0
File tidak ada di disk    : 0
Citra rusak               : 0
Cloudy label lain dihapus : 0
Dibuang cuaca ganda       : 0
Dibuang tanpa label cuaca : 1 ['train_24448']
Total data bersih         : 40478
Metode split              : fallback lokal: iterative stratification dua tahap plus penyeimbangan tukar (swap) penekan deviasi proporsi label (skmultilearn tidak tersedia)
Deviasi proporsi maksimum : 0.001367


In [3]:
dist = pd.read_csv(PROC / "label_distribution.csv")
print(dist.to_string(index=False))

            label  sebelum_cleaning  sesudah_cleaning  proporsi_sesudah
      agriculture             12315             12315          0.304239
   artisinal_mine               339               339          0.008375
      bare_ground               862               862          0.021296
         blooming               332               332          0.008202
        blow_down                98                98          0.002421
            clear             28431             28431          0.702382
           cloudy              2089              2089          0.051608
conventional_mine               100               100          0.002470
      cultivation              4477              4477          0.110603
       habitation              3660              3660          0.090419
             haze              2697              2697          0.066629
    partly_cloudy              7261              7261          0.179381
          primary             37513             37513          0

**Apa, mengapa, bagaimana untuk cleaning:**

- **Apa:** berkas citra diperiksa integritasnya satu per satu, lalu empat kebijakan label diterapkan: citra rusak atau hilang dibuang; label cuaca lebih dari satu dibuang; tanpa label cuaca dibuang; citra cloudy yang membawa label lain dipangkas menjadi cloudy saja.
- **Mengapa:** citra rusak akan menghentikan pelatihan di tengah jalan; cuaca pada satu potret hanya mungkin satu keadaan sehingga cuaca ganda berarti anotasi ambigu; tanpa label cuaca berarti di luar skema dataset; pada citra cloudy isi lahan tertutup awan sehingga label lahannya tidak bisa dipercaya, tetapi citranya tetap contoh cuaca yang sah.
- **Bagaimana:** pemeriksaan citra memakai Pillow dua tahap (verify lalu load) dan berjalan paralel; semua tindakan dicatat di cleaning_log.json. Hasil pada data asli: hanya satu baris dibuang (train_24448, tanpa label cuaca), data bersih 40.478 dari 40.479.

## 2. Feature engineering

Dua bentuk fitur disiapkan. Pertama, encoding label menjadi matriks biner, yaitu bentuk angka dari target. Kedua, fitur warna dan tekstur dari citra, yaitu masukan untuk model baseline dan advanced yang tidak membaca piksel mentah.

In [4]:
LABELS = ["agriculture", "artisinal_mine", "bare_ground", "blooming", "blow_down",
          "clear", "cloudy", "conventional_mine", "cultivation", "habitation",
          "haze", "partly_cloudy", "primary", "road", "selective_logging",
          "slash_burn", "water"]

def bangun_matriks_label(frame):
    return pd.DataFrame(
        [{lab: int(lab in set(tags.split())) for lab in LABELS} for tags in frame["tags"]],
        index=frame["image_name"].values)

df_bersih = pd.read_csv(PROC / "train_v2_clean.csv", dtype=str)
Y = bangun_matriks_label(df_bersih)
print("Bentuk matriks label (citra, label):", Y.shape)
print(Y.head())

Bentuk matriks label (citra, label): (40478, 17)
         agriculture  artisinal_mine  bare_ground  blooming  blow_down  clear  \
train_0            0               0            0         0          0      0   
train_1            1               0            0         0          0      1   
train_2            0               0            0         0          0      1   
train_3            0               0            0         0          0      1   
train_4            1               0            0         0          0      1   

         cloudy  conventional_mine  cultivation  habitation  haze  \
train_0       0                  0            0           0     1   
train_1       0                  0            0           0     0   
train_2       0                  0            0           0     0   
train_3       0                  0            0           0     0   
train_4       0                  0            0           1     0   

         partly_cloudy  primary  road  selective_

In [5]:
import numpy as np
from PIL import Image

def fitur_warna_tekstur(path):
    a = np.asarray(Image.open(path).convert("RGB"), dtype=np.float64)
    r, g, b = a[:, :, 0], a[:, :, 1], a[:, :, 2]
    abu = a.mean(axis=2)
    return {
        "mean_r": r.mean(), "mean_g": g.mean(), "mean_b": b.mean(),
        "std_r": r.std(), "std_g": g.std(), "std_b": b.std(),
        "kecerahan": abu.mean(), "kontras": abu.std(),
        "rasio_hijau": float((g > r).mean()),
    }

latih = pd.read_csv(PROC / "split_train.csv", dtype=str)
contoh = latih.head(100)
fitur = pd.DataFrame([fitur_warna_tekstur(RAW / "train-jpg" / f"{n}.jpg")
                      for n in contoh["image_name"]])
print("Fitur terhitung untuk", len(fitur), "citra contoh dari data latih:")
print(fitur.head())
print()
print(fitur.describe().loc[["mean", "std"]].round(2))

Fitur terhitung untuk 100 citra contoh dari data latih:
      mean_r      mean_g      mean_b      std_r      std_g      std_b  \
0  91.787384  109.347000  103.609146   4.262149   3.075204   2.588401   
1  65.639862   76.497375   64.301590  18.028302  17.253960   9.512584   
2  43.164474   58.039383   56.862000   6.040614   4.976010   3.360188   
3  51.411316   65.253769   53.972961   5.717531   4.917149   3.372893   
4  57.342377   45.253403   19.525055  19.869230  17.785704  11.227933   

    kecerahan    kontras  rasio_hijau  
0  101.581177   2.960481     1.000000  
1   68.812943  14.734115     0.975296  
2   52.688619   4.483417     0.999924  
3   56.879349   4.373292     0.999893  
4   40.706945  16.177663     0.000076  

      mean_r  mean_g  mean_b  std_r  std_g  std_b  kecerahan  kontras  \
mean   80.31   86.70   76.36  12.30   9.66   7.75      81.12     9.68   
std    38.62   33.43   32.47   9.46   7.24   6.44      34.45     7.62   

      rasio_hijau  
mean         0.78  
std 

**Apa, mengapa, bagaimana untuk feature engineering:**

- **Apa:** label diubah menjadi matriks biner 17 kolom (satu berarti label ada), dan dari tiap citra dihitung fitur warna (rata rata dan sebaran tiap kanal) serta tekstur sederhana (kecerahan, kontras, rasio piksel hijau dominan).
- **Mengapa:** model dan perhitungan statistik bekerja dengan angka, bukan kata. Model klasik seperti Logistic Regression dan Random Forest tidak bisa membaca piksel mentah dengan baik, jadi citra harus diterjemahkan menjadi angka ringkasan yang bermakna: hutan sehat cenderung hijau dan bertekstur kasar, lahan terbuka cenderung cerah dan rata.
- **Bagaimana:** matriks label dibangun dari data bersih untuk seluruh citra. Fitur warna tekstur di notebook ini dicontohkan pada 100 citra data latih; ekstraksi penuh untuk seluruh data latih dilakukan pada tahap pemodelan baseline. Sesuai checklist, scaler untuk fitur ini hanya boleh di-fit pada data training, tidak pada validasi dan uji, supaya tidak ada informasi yang bocor dari data penilaian.

## 3. Transformasi data

Keputusan transformasi untuk proyek ini:

- **Encoding:** sudah dilakukan sebagai matriks biner multi-label pada bagian 2. Ini adalah encoding yang benar untuk multi-label; one-hot satu kelas tidak berlaku karena satu citra punya banyak label.
- **Scaling piksel:** nilai piksel 0 sampai 255 diskalakan menjadi 0 sampai 1 saat citra dibaca model, dicontohkan pada sel di bawah. Untuk fitur warna tekstur, scaling dengan scaler yang di-fit hanya pada data training dilakukan pada tahap pemodelan.
- **Tokenisasi:** tidak berlaku, karena data ini citra, bukan teks. Satu satunya data teks adalah nama citra dan kata label, yang sudah ditangani encoding.
- **Augmentasi:** tidak dilakukan pada tahap persiapan. Augmentasi citra (balik horizontal, putar, potong acak) hanya akan diterapkan pada data training saat pelatihan deep learning, tidak pernah pada validasi dan uji, supaya data penilaian tetap asli.

In [6]:
a = np.asarray(Image.open(RAW / "train-jpg" / f"{latih['image_name'].iloc[0]}.jpg"), dtype=np.float64)
print("Sebelum scaling: minimum", a.min(), "| maksimum", a.max(), "| rata rata", round(float(a.mean()), 2))
a2 = a / 255.0
print("Sesudah scaling : minimum", a2.min(), "| maksimum", a2.max(), "| rata rata", round(float(a2.mean()), 4))

Sebelum scaling: minimum 0.0 | maksimum 181.0 | rata rata 115.06
Sesudah scaling : minimum 0.0 | maksimum 0.7098039215686275 | rata rata 0.4512


## 4. Splitting train, validation, test

Pembagian dilakukan skrip persiapan data dengan iterative stratification dan seed tetap 42: label paling langka ditempatkan lebih dulu, lalu proporsi tiap label dijaga sedekat mungkin dengan proporsi keseluruhan, ditambah penyeimbangan dengan menukar posisi sampel. Sel di bawah memverifikasi hasilnya: ukuran tiap bagian, tidak ada citra yang muncul di dua bagian, dan deviasi proporsi label.

In [7]:
splits = {nama: pd.read_csv(PROC / f"split_{nama}.csv", dtype=str) for nama in ["train", "val", "test"]}
for nama, f in splits.items():
    print(nama, ":", len(f), "citra")
nama_train = set(splits["train"]["image_name"]); nama_val = set(splits["val"]["image_name"]); nama_test = set(splits["test"]["image_name"])
print("Irisan train-val:", len(nama_train & nama_val), "| train-test:", len(nama_train & nama_test), "| val-test:", len(nama_val & nama_test))

def hitung(frame):
    c = {}
    for t in frame["tags"]:
        for lab in set(t.split()):
            c[lab] = c.get(lab, 0) + 1
    return c

total = hitung(df_bersih); n_total = len(df_bersih)
dev_maks = 0.0
for f in splits.values():
    c = hitung(f); n = len(f)
    for lab in LABELS:
        dev_maks = max(dev_maks, abs(c.get(lab, 0) / n - total.get(lab, 0) / n_total))
print("Deviasi proporsi label maksimum (hitung ulang):", round(dev_maks, 6))

train : 28335 citra
val : 6071 citra
test : 6072 citra
Irisan train-val: 0 | train-test: 0 | val-test: 0


Deviasi proporsi label maksimum (hitung ulang): 0.001367


**Apa, mengapa, bagaimana untuk splitting:**

- **Apa:** data bersih 40.478 citra dibagi menjadi latih 28.335 (70 persen), validasi 6.071 (15 persen), dan uji 6.072 (15 persen).
- **Mengapa:** data latih untuk melatih model, validasi untuk menyetel model dan threshold, uji disimpan untuk penilaian akhir yang jujur. Pembagian acak murni berbahaya untuk label langka: blow_down hanya 98 contoh dan bisa kosong di data uji bila dibagi sembarangan.
- **Bagaimana:** iterative stratification dua tahap dengan seed 42, ditambah penyeimbangan tukar. Keberhasilannya terukur: deviasi proporsi label maksimum hanya 0,001367. Data uji tidak disentuh untuk penyetelan apa pun dan disimpan sampai tahap evaluasi.

## 5. Data balancing

**Keputusan: tidak dilakukan oversampling atau undersampling pada tahap persiapan.** Alasannya, menyeimbangkan data pada tahap ini akan mengubah distribusi asli yang menjadi dasar evaluasi, dan label langka seperti blow_down (98 contoh) akan tetap langka informasinya walau digandakan. Penanganan ketidakseimbangan dilakukan pada tahap pemodelan dengan cara yang tidak mengubah data penilaian: metrik F2 yang memprioritaskan recall, pembobotan kelas pada fungsi loss, dan threshold per label yang disetel di data validasi. Bila nanti diputuskan melakukan balancing, sesuai aturan ia hanya boleh diterapkan pada data training, tidak pada validasi dan uji. Sel di bawah menunjukkan distribusi alami label inti di tiap bagian sebagai bukti tidak ada balancing yang diterapkan.

In [8]:
inti = ["slash_burn", "selective_logging", "artisinal_mine", "conventional_mine", "blow_down"]
for nama, f in splits.items():
    c = hitung(f)
    print(nama, {lab: c.get(lab, 0) for lab in inti})

train {'slash_burn': 146, 'selective_logging': 238, 'artisinal_mine': 237, 'conventional_mine': 70, 'blow_down': 69}
val {'slash_burn': 32, 'selective_logging': 51, 'artisinal_mine': 51, 'conventional_mine': 15, 'blow_down': 14}
test {'slash_burn': 31, 'selective_logging': 51, 'artisinal_mine': 51, 'conventional_mine': 15, 'blow_down': 15}


## 6. Ringkasan Apa Mengapa Bagaimana

| Langkah | Apa | Mengapa | Bagaimana |
|---|---|---|---|
| Cek integritas citra | Semua citra dibuka dan dibaca pikselnya | Berkas rusak menghentikan pelatihan di tengah jalan | Pillow verify lalu load, berjalan paralel, hasil nol rusak |
| Cleaning label | Empat kebijakan aturan dataset diterapkan | Menghapus anotasi ambigu dan di luar skema | Rusak dan hilang dibuang, cuaca ganda dibuang, tanpa cuaca dibuang, cloudy dipangkas |
| Encoding label | Tags menjadi matriks biner 17 kolom | Model bekerja dengan angka; multi-label butuh banyak kolom | Satu berarti label hadir, mengikuti urutan LABELS baku |
| Fitur warna tekstur | Angka ringkasan warna dan permukaan citra | Model klasik tidak membaca piksel mentah | Rata rata dan sebaran kanal, kecerahan, kontras, rasio hijau |
| Scaling | Piksel 0 sampai 255 menjadi 0 sampai 1 | Nilai besar memperlambat dan mengacaukan pelatihan | Pembagian 255 saat citra dibaca; scaler fitur di-fit hanya di data latih |
| Augmentasi | Tidak dilakukan pada persiapan | Data penilaian harus tetap asli | Nanti hanya pada data training saat pelatihan deep learning |
| Splitting | Latih 70, validasi 15, uji 15 persen | Penilaian akhir harus jujur dan label langka terwakili | Iterative stratification, seed 42, deviasi maksimum 0,001367 |
| Balancing | Tidak dilakukan pada persiapan | Distribusi asli adalah dasar evaluasi yang sah | Ditangani lewat F2, bobot kelas, dan threshold; bila dilakukan hanya di data training |

**Checklist penyelesaian tahap ini:**

- Scaler dan encoder di-fit hanya pada data training: dipatuhi, encoding label bersifat tetap tanpa fitting, dan scaler fitur dinyatakan hanya di-fit pada data latih.
- Test set disimpan dan tidak disentuh sampai tahap evaluasi akhir: dipatuhi, data uji tidak dipakai menyetel apa pun.
- Jumlah sampel tiap split tercatat: dipatuhi, 28.335 latih, 6.071 validasi, 6.072 uji, tercatat di cleaning_log.json dan notebook ini.
- Setiap langkah preprocessing punya alasan tertulis: dipatuhi, lihat ringkasan di atas.